In [16]:
import sys
import os

scripts_path = os.path.abspath(os.path.join(os.getcwd(), '..', 'scripts'))
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)


In [17]:
# TESTES 1 e 2 - TRADE-OFFS no README.md

In [18]:
import pandas as pd
from api_requests import baixar_e_descompactar

In [19]:
from api_requests import baixar_e_descompactar

api_url = [
    'https://dadosabertos.ans.gov.br/FTP/PDA/demonstracoes_contabeis/2025/1T2025.zip',
    'https://dadosabertos.ans.gov.br/FTP/PDA/demonstracoes_contabeis/2025/2T2025.zip',
    'https://dadosabertos.ans.gov.br/FTP/PDA/demonstracoes_contabeis/2025/3T2025.zip',
    #'https://dadosabertos.ans.gov.br/FTP/PDA/operadoras_acreditadas/operadoras_acreditadas.csv',
    'https://dadosabertos.ans.gov.br/FTP/PDA/operadoras_de_plano_de_saude_ativas/Relatorio_cadop.csv',
    'https://dadosabertos.ans.gov.br/FTP/PDA/operadoras_de_plano_de_saude_canceladas/Relatorio_cadop_canceladas.csv',
    #'https://dadosabertos.ans.gov.br/FTP/PDA/operadoras_e_prestadores_nao_hospitalares/operadoras_e_prestadores_nao_hospitalares.zip'
]

baixar_e_descompactar(api_url)

Baixando 1T2025.zip...
Descompactando 1T2025.zip
Baixando 2T2025.zip...
Descompactando 2T2025.zip
Baixando 3T2025.zip...
Descompactando 3T2025.zip
Baixando Relatorio_cadop.csv...
Baixando Relatorio_cadop_canceladas.csv...


In [20]:
import pandas as pd
import os

DATA_DIR = os.path.join('..', 'data')

#apenas arquivos dos trimestres
paths = [
    os.path.join(DATA_DIR, f)
    for f in os.listdir(DATA_DIR)
    if f.endswith(".csv") and f.startswith(("1T", "2T", "3T"))
]

def le_exibe_colunas(path):
    chunk = next(pd.read_csv(path, sep=';', encoding='utf-8', chunksize=100_000))
    print(chunk.columns.tolist())
    
    
for path in paths:
    print(f"\nArquivo: {path}")
    le_exibe_colunas(path)

def trimestre_filtrados(path, valor, coluna='DESCRICAO'):
    for chunk in pd.read_csv(path, sep=';', encoding='utf-8', chunksize=100_000):
        mask = (
            chunk[coluna]
            .astype(str)
            .str.strip()
            .str.upper()
            .str.contains(valor.upper(), na=False, regex=True)
        )
        if mask.any():
            yield chunk.loc[mask]
            
df_eventos = pd.concat(
    (
        chunk
        for path in paths
        for chunk in trimestre_filtrados(path, r'(?=.*EVENT)(?=.*SINISTR)')
    ),
    ignore_index=True
)

df_eventos.head()




Arquivo: ../data/1T2025.csv
['DATA', 'REG_ANS', 'CD_CONTA_CONTABIL', 'DESCRICAO', 'VL_SALDO_INICIAL', 'VL_SALDO_FINAL']

Arquivo: ../data/2T2025.csv
['DATA', 'REG_ANS', 'CD_CONTA_CONTABIL', 'DESCRICAO', 'VL_SALDO_INICIAL', 'VL_SALDO_FINAL']

Arquivo: ../data/3T2025.csv
['DATA', 'REG_ANS', 'CD_CONTA_CONTABIL', 'DESCRICAO', 'VL_SALDO_INICIAL', 'VL_SALDO_FINAL']


,DATA,REG_ANS,CD_CONTA_CONTABIL,DESCRICAO,VL_SALDO_INICIAL,VL_SALDO_FINAL
0,2025-01-01,316849,131719011,Depósitos Judiciais - Eventos / Sinistros,"289349,17","292907,23"
1,2025-01-01,316849,21111203,Provisão de Eventos/Sinistros a Liquidar para ...,0,0
2,2025-01-01,316849,23111202,Provisão de Eventos/Sinistros a Liquidar para ...,"203169,04","199868,05"
3,2025-01-01,316849,231112022,Provisão de Eventos/Sinistros a Liquidar para ...,"203169,04","199868,05"
4,2025-01-01,316903,231111021,Provisão de Eventos/Sinistros a Liquidar para ...,25535,25535


In [21]:
# Baixei todos os arquivos de operadoras inicialmente. No desafio 1.3 é necessario criar um .csv com colunas: 
# CNPJ , RazaoSocial , Trimestre , Ano , ValorDespesaTotal. Através da análise dos arquivos, notei que os arquivos de operadoras de plano de saúde são
# os que devem ser utilizados na análise, pois operadoras não hospitalares e operadoras acreditadas não possuem informações necessárias. Colunas das mesmas:
# ['Reg ANS', 'Operadora', 'Nivel_Acreditacao', 'Inicio_Validade', 'Fim_Validade', 'Prazo_validade', 'Resolucao_Normativa', 'Entidade_Acreditadora', 'Reacreditada', 'Data_da_Primeira_Acreditacao', 'Total de Anos de Acredita\x87Æo*']
# ['REGISTRO_OPERADORA', 'NM_OPERADORA', 'GR_MODALIDADE', 'ID_ESTABELECIMENTO_SAUDE', 'CD_CNPJ_ESTB_SAUDE', 'CD_CNES', 'NM_ESTABELECIMENTO_SAUDE', 'DE_CLAS_ESTB_SAUDE', 'DE_TIPO_PRESTADOR', 'LG_URGENCIA_EMERGENCIA', 'DE_TIPO_CONTRATO', 'DE_DISPONIBILIDADE', 'CD_MUNICIPIO', 'NM_MUNICIPIO', 'SG_UF', 'NM_REGIAO', 'DT_VINCULO_OPERADORA_INICIO', 'DT_VINCULO_OPERADORA_FIM', 'COMPETENCIA', 'DT_ATUALIZACAO']

In [22]:
operadoras = [
    os.path.join(DATA_DIR, f)
    for f in os.listdir(DATA_DIR)
    if f.startswith(("Relatorio")) and f.endswith(".csv")    
]

for operador in operadoras:
    print(f"Arquivo: {operador} ")
    le_exibe_colunas(operador)


Arquivo: ../data/Relatorio_cadop.csv 
['REGISTRO_OPERADORA', 'CNPJ', 'Razao_Social', 'Nome_Fantasia', 'Modalidade', 'Logradouro', 'Numero', 'Complemento', 'Bairro', 'Cidade', 'UF', 'CEP', 'DDD', 'Telefone', 'Fax', 'Endereco_eletronico', 'Representante', 'Cargo_Representante', 'Regiao_de_Comercializacao', 'Data_Registro_ANS']
Arquivo: ../data/Relatorio_cadop_canceladas.csv 
['REGISTRO_OPERADORA', 'CNPJ', 'Razao_Social', 'Nome_Fantasia', 'Modalidade', 'Logradouro', 'Numero', 'Complemento', 'Bairro', 'Cidade', 'UF', 'CEP', 'DDD', 'Telefone', 'Fax', 'Endereco_eletronico', 'Representante', 'Cargo_Representante', 'Regiao_de_Comercializacao', 'Data_Registro_ANS', 'Data_Descredenciamento', 'Motivo_do_Descredenciamento']


In [23]:


def trata_relatorio_cadop(filepath):
    df = pd.read_csv(filepath, sep=';', encoding='utf-8', dtype=str)
    
    print(f"Registros originais: {len(df)}")
    
   
    campos_obrigatorios = ['REGISTRO_OPERADORA', 'CNPJ', 'Razao_Social']
    for campo in campos_obrigatorios:
        df = df[df[campo].notna() & (df[campo].str.strip() != '')]
    print(f"Após remover NULLs em campos obrigatórios: {len(df)}")
    
    
    df['CNPJ'] = df['CNPJ'].str.replace(r'\D', '', regex=True) 
    df = df[df['CNPJ'].str.len() == 14]
    df = df[df['CNPJ'].apply(cnpj_tool.validate)]
    print(f"Após validar CNPJs: {len(df)}")
    
   
    df['REGISTRO_OPERADORA'] = pd.to_numeric(df['REGISTRO_OPERADORA'], errors='coerce')
    df = df[df['REGISTRO_OPERADORA'].notna()]
    df['REGISTRO_OPERADORA'] = df['REGISTRO_OPERADORA'].astype(int).astype(str)
    
    
    df['DDD'] = df['DDD'].str.replace(r'\D', '', regex=True)
    
    
    df['CEP'] = df['CEP'].str.replace(r'\D', '', regex=True)
    
    
    df['Regiao_de_Comercializacao'] = pd.to_numeric(df['Regiao_de_Comercializacao'], errors='coerce')
    df['Regiao_de_Comercializacao'] = df['Regiao_de_Comercializacao'].fillna(0).astype(int).astype(str)
    
    
   
    df['Data_Registro_ANS'] = pd.to_datetime(df['Data_Registro_ANS'], errors='coerce')
    registros_data_invalida = df['Data_Registro_ANS'].isna().sum()
    print(f"Registros com data inválida: {registros_data_invalida}")
    df = df[df['Data_Registro_ANS'].notna()]
    df['Data_Registro_ANS'] = df['Data_Registro_ANS'].dt.strftime('%Y-%m-%d')
    
    
   
    campos_texto = ['Razao_Social', 'Nome_Fantasia', 'Modalidade', 'Logradouro', 
                    'Numero', 'Complemento', 'Bairro', 'Cidade', 'Representante', 
                    'Cargo_Representante', 'Endereco_eletronico']
    for campo in campos_texto:
        if campo in df.columns:
            df[campo] = df[campo].fillna('').str.strip()
    
  
    df['UF'] = df['UF'].str.upper().str.strip()
    df = df[df['UF'].str.len() == 2]
    
    return df


cadop_path = os.path.join(DATA_DIR, 'Relatorio_cadop.csv')
df_cadop_tratado = trata_relatorio_cadop(cadop_path)


df_cadop_tratado.to_csv(cadop_path, index=False, sep=';', encoding='utf-8')
df_cadop_tratado.head()

Registros originais: 1110
Após remover NULLs em campos obrigatórios: 1110
Após validar CNPJs: 1110
Registros com data inválida: 0


,REGISTRO_OPERADORA,CNPJ,Razao_Social,Nome_Fantasia,Modalidade,Logradouro,Numero,Complemento,Bairro,Cidade,UF,CEP,DDD,Telefone,Fax,Endereco_eletronico,Representante,Cargo_Representante,Regiao_de_Comercializacao,Data_Registro_ANS
0,419761,19541931000125,18 DE JULHO ADMINISTRADORA DE BENEFÍCIOS LTDA,,Administradora de Benefícios,RUA CAPITÃO MEDEIROS DE REZENDE,274,,PRAÇA DA BANDEIRA,Além Paraíba,MG,36660000,32,34624649,NaN,contabilidade@cbnassessoria.com.br,LUIZ HENRIQUE MARENDINO GONÇALVES,SÓCIO ADMINISTRADOR,6,2015-05-19
1,421545,22869997000153,2B ODONTOLOGIA OPERADORA DE PLANOS ODONTOLÓGIC...,,Odontologia de Grupo,RUA CATÃO,128,SALA 126,VILA ROMANA,São Paulo,SP,05049000,11,34415852,NaN,labmarisol@gmail.com,MARISOL BECHELLI,SÓCIO ADMINISTRADORA,4,2019-06-13
2,421421,27452545000195,2CARE OPERADORA DE SAÚDE LTDA.,,Medicina de Grupo,RUA: BERNARDINO DE CAMPOS,230,1º ANDAR,CENTRO,Campinas,SP,13010151,19,37901224,NaN,ans.plano@hospitalcare.com.br,RODRIGO PINHO RIBEIRO,REPRESENTANTE,5,2018-10-09
3,418030,13138885000131,A.P.S. ADMINISTRADORA DE BENEFÍCOS LTDA.,A.P.S. SAÚDE.,Administradora de Benefícios,RUA VOLUNTÁRIOS DA PÁTRIA,2525,CONJUNTO 143 - SALA 01,SANTANA,São Paulo,SP,02401000,11,45223468,NaN,diretoria@apssaude.com.br,PERCÍVEL GAETA,SóCIO-ADMINISTRADOR E REPRESEN,4,2011-05-05
4,314668,17505793000101,ABERTTA SAÚDE - ASSOCIAÇÃO BENEFICENTE DOS EMP...,ABERTTA SAÚDE,Autogestão,AV. BERNARDO MONTEIRO,831,"Subsolo, 2º andar e 3º andar",SANTA EFIGÊNIA,Belo Horizonte,MG,30150281,31,32484300,32484377,abertta.ans@arcelormittal.com.br,WERNER DUARTE DALLA,Diretor Presidente,4,1998-12-28


In [24]:
#gera df operadoras ativas / canceladas em 2025

In [25]:
dfs_operadoras = []
for operador in operadoras:
    df = pd.read_csv(operador, sep=';', encoding='utf-8')
    if 'Data_Descredenciamento' in df.columns:
        df['Data_Descredenciamento'] = pd.to_datetime(df['Data_Descredenciamento'], errors='coerce')
        df = df[df['Data_Descredenciamento'].isna() | (df['Data_Descredenciamento'].dt.year >= 2025)]
        dfs_operadoras.append(df)
    else:
        dfs_operadoras.append(df)

dfs_operadoras = pd.concat(dfs_operadoras, ignore_index=True)

dfs_operadoras.head()


,REGISTRO_OPERADORA,CNPJ,Razao_Social,Nome_Fantasia,Modalidade,Logradouro,Numero,Complemento,Bairro,Cidade,...,DDD,Telefone,Fax,Endereco_eletronico,Representante,Cargo_Representante,Regiao_de_Comercializacao,Data_Registro_ANS,Data_Descredenciamento,Motivo_do_Descredenciamento
0,419761,19541931000125,18 DE JULHO ADMINISTRADORA DE BENEFÍCIOS LTDA,NaN,Administradora de Benefícios,RUA CAPITÃO MEDEIROS DE REZENDE,274,NaN,PRAÇA DA BANDEIRA,Além Paraíba,...,32.0,34624649.0,NaN,contabilidade@cbnassessoria.com.br,LUIZ HENRIQUE MARENDINO GONÇALVES,SÓCIO ADMINISTRADOR,6.0,2015-05-19,NaT,NaN
1,421545,22869997000153,2B ODONTOLOGIA OPERADORA DE PLANOS ODONTOLÓGIC...,NaN,Odontologia de Grupo,RUA CATÃO,128,SALA 126,VILA ROMANA,São Paulo,...,11.0,34415852.0,NaN,labmarisol@gmail.com,MARISOL BECHELLI,SÓCIO ADMINISTRADORA,4.0,2019-06-13,NaT,NaN
2,421421,27452545000195,2CARE OPERADORA DE SAÚDE LTDA.,NaN,Medicina de Grupo,RUA: BERNARDINO DE CAMPOS,230,1º ANDAR,CENTRO,Campinas,...,19.0,37901224.0,NaN,ans.plano@hospitalcare.com.br,RODRIGO PINHO RIBEIRO,REPRESENTANTE,5.0,2018-10-09,NaT,NaN
3,418030,13138885000131,A.P.S. ADMINISTRADORA DE BENEFÍCOS LTDA.,A.P.S. SAÚDE.,Administradora de Benefícios,RUA VOLUNTÁRIOS DA PÁTRIA,2525,CONJUNTO 143 - SALA 01,SANTANA,São Paulo,...,11.0,45223468.0,NaN,diretoria@apssaude.com.br,PERCÍVEL GAETA,SóCIO-ADMINISTRADOR E REPRESEN,4.0,2011-05-05,NaT,NaN
4,314668,17505793000101,ABERTTA SAÚDE - ASSOCIAÇÃO BENEFICENTE DOS EMP...,ABERTTA SAÚDE,Autogestão,AV. BERNARDO MONTEIRO,831,"Subsolo, 2º andar e 3º andar",SANTA EFIGÊNIA,Belo Horizonte,...,31.0,32484300.0,32484377.0,abertta.ans@arcelormittal.com.br,WERNER DUARTE DALLA,Diretor Presidente,4.0,1998-12-28,NaT,NaN


In [26]:
def gera_consolidado(df):
    df['Trimestre'] = df['DATA'].str.slice(5,7).map({'01':'1T','04':'2T','07':'3T'})
    df['Ano'] = df['DATA'].str.slice(0,4)
    df['VL_SALDO_FINAL'] = pd.to_numeric(df['VL_SALDO_FINAL'].str.replace(',', '.', regex=False), errors='coerce')
    df['VL_SALDO_INICIAL'] = pd.to_numeric(df['VL_SALDO_INICIAL'].str.replace(',', '.', regex=False), errors='coerce')
    df['ValorDespesas'] = (
    (df['VL_SALDO_FINAL'] - df['VL_SALDO_INICIAL']) * 100).round().astype('Int64')

    df_consolidado = df[['CNPJ', 'Razao_Social', 'Trimestre', 'Ano', 'ValorDespesas']]

    return df_consolidado

df_final = (
    df_eventos.merge(
    dfs_operadoras, 
    left_on='REG_ANS', 
    right_on='REGISTRO_OPERADORA', 
    how='inner'   
    )
    .pipe(gera_consolidado)
)


df_final.head(10)

,CNPJ,Razao_Social,Trimestre,Ano,ValorDespesas
0,42465310000121,TELOS - FUNDAÇÃO EMBRATEL DE SEGURIDADE SOCIAL,1T,2025,355806
1,42465310000121,TELOS - FUNDAÇÃO EMBRATEL DE SEGURIDADE SOCIAL,1T,2025,0
2,42465310000121,TELOS - FUNDAÇÃO EMBRATEL DE SEGURIDADE SOCIAL,1T,2025,-330099
3,42465310000121,TELOS - FUNDAÇÃO EMBRATEL DE SEGURIDADE SOCIAL,1T,2025,-330099
4,93507895000136,POLIMÉDICA SAÚDE SOCIEDADE SIMPLES LTDA,1T,2025,0
5,93507895000136,POLIMÉDICA SAÚDE SOCIEDADE SIMPLES LTDA,1T,2025,12467797
6,93507895000136,POLIMÉDICA SAÚDE SOCIEDADE SIMPLES LTDA,1T,2025,12487667
7,93507895000136,POLIMÉDICA SAÚDE SOCIEDADE SIMPLES LTDA,1T,2025,10222376
8,93507895000136,POLIMÉDICA SAÚDE SOCIEDADE SIMPLES LTDA,1T,2025,10222376
9,93507895000136,POLIMÉDICA SAÚDE SOCIEDADE SIMPLES LTDA,1T,2025,10222376


In [27]:
# FINAL TESTE 1 -> INICIO TESTE 2 
# Encontrados arquivos inconsistentes, com despesas negativas, zeradas, etc. 
# CNPJs duplicado também, trimestres aparentemente normais. 
# Solução: Corrigir. Já pegando o gancho do teste 2.1, vou validar valores 0/-,
# verificar trimestres e validar, corrigir cnpjs duplicados e com razoes sociais diferentes 
#  e aplicar uma validação de formato. Também verificar razão social não vazia.
#  TRADE-OFF DA VALIDAÇÃO: NO README.md

In [28]:
%pip install pandas validate-docbr

from validate_docbr import CNPJ

cnpj_tool = CNPJ()

def valida_valores_negativos_e_zeros(df):
    df_validado = df[(df['ValorDespesas'] > 0)]
    return df_validado

def valida_trimestres(df):
    trimestres_validos = {'1T', '2T', '3T'}
    df_validado = df[df['Trimestre'].isin(trimestres_validos)]
    return df_validado

def valida_cnpjs(df):
    
    #primeiro valida formato
    df['CNPJ'] = df['CNPJ'].astype(str)
    df = df[df['CNPJ'].apply(cnpj_tool.validate)]
    
    #valida duplicatas
    df = df.drop_duplicates(subset=['CNPJ', 'Trimestre'], keep='first')
    
    #valida razão social difernente para mesmo CNPJ
    df = df.sort_values(by=['CNPJ', 'Razao_Social'])
    df = df.drop_duplicates(subset=['CNPJ', 'Trimestre'], keep='first')
    
    #valida razao social vazia
    df = df[df['Razao_Social'].notna() & (df['Razao_Social'].str.strip() != '')]
    
    return df

df_final = (
    df_final.pipe(valida_valores_negativos_e_zeros)
            .pipe(valida_trimestres)
            .pipe(valida_cnpjs)
)

print(df_final.dtypes)
print(df_final.head())
csv_path = os.path.join(DATA_DIR, 'consolidados_despesas.csv')
zip_path = os.path.join(DATA_DIR, 'consolidados_despesas.zip')

df_final['ValorDespesas'] = df_final['ValorDespesas'].round(2)
df_final.to_csv(csv_path, index=False, sep=';', encoding='utf-8')

with open(csv_path, 'rb') as f_in, open(zip_path, 'wb') as f_out:
    f_out.write(f_in.read())
    

Note: you may need to restart the kernel to use updated packages.
CNPJ               str
Razao_Social       str
Trimestre          str
Ano                str
ValorDespesas    Int64
dtype: object
                  CNPJ                                       Razao_Social  \
53970   10219897000100  UNIMED OESTE DO PARÁ - COOPERATIVA DE TRABALHO...   
103246  10219897000100  UNIMED OESTE DO PARÁ - COOPERATIVA DE TRABALHO...   
117994  10219897000100  UNIMED OESTE DO PARÁ - COOPERATIVA DE TRABALHO...   
13576   10364053000145  ODONTOLIVE OPERADORA DE PLANOS ODONTOLÓGICOS L...   
78223   10364053000145  ODONTOLIVE OPERADORA DE PLANOS ODONTOLÓGICOS L...   

       Trimestre   Ano  ValorDespesas  
53970         1T  2025       44776509  
103246        2T  2025       11813206  
117994        3T  2025     2362858380  
13576         1T  2025         258124  
78223         2T  2025        2219128  


In [29]:
df_operadoras_ativas = (
    pd.read_csv(os.path.join(DATA_DIR, 'Relatorio_cadop.csv'), sep=';', encoding='utf-8')   
)



def trata_cpnjs_duplicados(df1):
    df1['CNPJ'] = df1['CNPJ'].astype(str)
    
    df1['Data_Registro_ANS'] = pd.to_datetime(df1['Data_Registro_ANS'], errors='coerce')
    df_tratado = df1.sort_values(by=['Data_Registro_ANS']).drop_duplicates(subset=['CNPJ'], keep='last')   
    
    return df_tratado

df_operadoras_ativas_tratado = trata_cpnjs_duplicados(df_operadoras_ativas)

df_tratado = pd.merge(
    left=df_final, 
    right=df_operadoras_ativas_tratado, 
    on='CNPJ',
    how='inner' 
)

#lembrar trade-off 2.2

df_tratado = df_tratado.rename(columns={'Razao_Social_x': 'Razao_Social'})
df_tratado = df_tratado.rename(columns={'REGISTRO_OPERADORA': 'RegistroANS'})
df_tratado = df_tratado[['CNPJ', 'Razao_Social', 'Trimestre', 'Ano', 'ValorDespesas', 'RegistroANS', 'Modalidade', 'UF']]


df_tratado.head()

,CNPJ,Razao_Social,Trimestre,Ano,ValorDespesas,RegistroANS,Modalidade,UF
0,10219897000100,UNIMED OESTE DO PARÁ - COOPERATIVA DE TRABALHO...,1T,2025,44776509,362140,Cooperativa Médica,PA
1,10219897000100,UNIMED OESTE DO PARÁ - COOPERATIVA DE TRABALHO...,2T,2025,11813206,362140,Cooperativa Médica,PA
2,10219897000100,UNIMED OESTE DO PARÁ - COOPERATIVA DE TRABALHO...,3T,2025,2362858380,362140,Cooperativa Médica,PA
3,10364053000145,ODONTOLIVE OPERADORA DE PLANOS ODONTOLÓGICOS L...,1T,2025,258124,417831,Odontologia de Grupo,SP
4,10364053000145,ODONTOLIVE OPERADORA DE PLANOS ODONTOLÓGICOS L...,2T,2025,2219128,417831,Odontologia de Grupo,SP


In [30]:
#Agrupar por Razao Social e UF

In [31]:
df_agregado = df_tratado.groupby(['Razao_Social', 'UF'], as_index=False)['ValorDespesas'].sum()
df_agregado['ValorDespesas'] = df_agregado['ValorDespesas'].round(2)

for operadora in df_agregado['Razao_Social'].unique():
    df_operadora = df_tratado[df_tratado['Razao_Social'] == operadora]
    soma_despesas_por_operadora = df_operadora['ValorDespesas'].sum().round(2)
    print(f'Operadora: {operadora} - {soma_despesas_por_operadora}')


for uf in df_agregado['UF'].unique():
    df_uf = df_tratado[df_tratado['UF'] == uf]
    soma_despesas_por_uf = df_uf['ValorDespesas'].sum().round(2)
    print(f'UF: {uf} - {soma_despesas_por_uf}')


# tempo curto, mas para fazer a media é agrupar por trimestres, apos somar tudo de cada
# desvio padrao usa std()

print(df_agregado.shape)
df_agregado = df_agregado.sort_values(by='ValorDespesas', ascending=False)
df_agregado.head(20)

#colocar trade off 2.3: optei por sort_values pois o volume de dados não é significativo
# caso fosse, seguir abordagem dos chunks novamente. 

output_dir = os.path.join('..', 'data')
os.makedirs(output_dir, exist_ok=True)

csv_path = os.path.join(output_dir, 'despesas_agregadas.csv')
zip_path = os.path.join(output_dir, 'Teste_Gabriel_Diniz.zip')

df_agregado.to_csv(csv_path, index=False, sep=';', encoding='utf-8')

compression_opts = dict(method='zip', archive_name='despesas_agregadas.csv')
df_agregado.to_csv(zip_path, index=False, sep=';', encoding='utf-8', compression=compression_opts)


Operadora: 2CARE OPERADORA DE SAÚDE LTDA. - 6494071611
Operadora: ABERTTA SAÚDE - ASSOCIAÇÃO BENEFICENTE DOS EMPREGADOS DA ARCELORMITTAL NO BRASIL - 10331612996
Operadora: AGROS - INSTITUTO UFV DE SEGURIDADE SOCIAL - 19786548
Operadora: ALICE OPERADORA LTDA. - 14868909996
Operadora: ALMA ODONTO OPERADORA DE PLANOS ODONTOLOGICOS LTDA - 23948929
Operadora: ALVORECER - ASSOCIAÇÃO DE SOCORROS MÚTUOS - 15349764568
Operadora: AMAZÔNIA PLANOS DE SAÚDE LTDA - 193469300
Operadora: AME VVIDA PLANOS DE SAUDE INTEGRADO LTDA. - 663331158
Operadora: AME-ASSISTÊNCIA MÉDICA A EMPRESAS LTDA - 246677642
Operadora: AMEPLAN ASSISTÊNCIA MÉDICA PLANEJADA LTDA - 485696524
Operadora: AMESC - ASSOCIAÇÃO MÉDICA ESPÍRITA CRISTÃ - 165424580
Operadora: AMHA SAUDE S/A - 5001092094
Operadora: AMHE MED ASSISTENCIA A SAUDE LTDA - EPP - 1744936315
Operadora: AMIL ASSISTÊNCIA MÉDICA INTERNACIONAL S.A. - 5280306718
Operadora: AMPARA ASSISTÊNCIA MÉDICA PARAÍSO LTDA - 1714477519
Operadora: AMPLA PLANOS DE SAUDE LTDA - 1725